# 01 — Single-Sérsic mock fit

This notebook validates the minimal BayMorph inference workflow on a synthetic single-Sérsic galaxy.

Workflow:

1. build a noiseless Sérsic mock;
2. define a constant noise map;
3. fit flux, effective radius, and Sérsic index with `DynestyFitter`;
4. inspect posterior summaries and residuals.

This notebook is intentionally small and does not use FITS I/O. FITS workflows are introduced in Notebook 02.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from baymorph import (
    DynestyFitter,
    Parameter,
    ParameterSet,
    SersicModel,
    make_constant_noise_map,
    plot_data_model_residual,
)

In [ ]:
from matplotlib.colors import LogNorm

BAYMORPH_IMAGE_CMAP = "turbo"


def _log_limits(
    *images,
    lower_percentile=0.5,
    upper_percentile=99.9,
    floor=1e-3,
):
    """Return robust positive limits for logarithmic image display."""
    positive_values = []

    for image in images:
        values = np.asarray(
            image,
            dtype=float,
        )
        values = values[
            np.isfinite(values)
            & (values > 0)
        ]

        if values.size > 0:
            positive_values.append(values)

    if len(positive_values) == 0:
        return floor, 1.0

    positive_values = np.concatenate(
        positive_values,
    )

    vmin = np.percentile(
        positive_values,
        lower_percentile,
    )
    vmax = np.percentile(
        positive_values,
        upper_percentile,
    )

    vmin = max(
        float(vmin),
        floor,
    )
    vmax = max(
        float(vmax),
        vmin * 10,
    )

    return vmin, vmax


def show_log_image(
    image,
    title,
    cmap=BAYMORPH_IMAGE_CMAP,
    colorbar_label="Flux",
):
    """Display an image using a robust logarithmic normalization."""
    vmin, vmax = _log_limits(
        image,
    )

    display_image = np.clip(
        np.asarray(
            image,
            dtype=float,
        ),
        vmin,
        vmax,
    )

    plt.figure(
        figsize=(6, 6),
    )

    plt.imshow(
        display_image,
        origin="lower",
        cmap=cmap,
        norm=LogNorm(
            vmin=vmin,
            vmax=vmax,
        ),
    )

    plt.colorbar(
        label=colorbar_label,
    )

    plt.title(
        f"{title} (log scale)",
    )

    plt.show()


def plot_log_data_model_residual(
    data,
    model,
    title="",
    cmap=BAYMORPH_IMAGE_CMAP,
):
    """Plot data and model on a log scale, with a linear residual panel."""
    data = np.asarray(
        data,
        dtype=float,
    )

    model = np.asarray(
        model,
        dtype=float,
    )

    residual = data - model

    vmin, vmax = _log_limits(
        data,
        model,
    )

    data_display = np.clip(
        data,
        vmin,
        vmax,
    )

    model_display = np.clip(
        model,
        vmin,
        vmax,
    )

    residual_limit = np.nanpercentile(
        np.abs(residual),
        99,
    )

    if not np.isfinite(residual_limit) or residual_limit == 0:
        residual_limit = 1.0

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15, 4),
    )

    data_im = axes[0].imshow(
        data_display,
        origin="lower",
        cmap=cmap,
        norm=LogNorm(
            vmin=vmin,
            vmax=vmax,
        ),
    )
    axes[0].set_title(
        "Data (log)"
    )
    fig.colorbar(
        data_im,
        ax=axes[0],
    )

    model_im = axes[1].imshow(
        model_display,
        origin="lower",
        cmap=cmap,
        norm=LogNorm(
            vmin=vmin,
            vmax=vmax,
        ),
    )
    axes[1].set_title(
        "Model (log)"
    )
    fig.colorbar(
        model_im,
        ax=axes[1],
    )

    residual_im = axes[2].imshow(
        residual,
        origin="lower",
        cmap="RdBu_r",
        vmin=-residual_limit,
        vmax=residual_limit,
    )
    axes[2].set_title(
        "Residual"
    )
    fig.colorbar(
        residual_im,
        ax=axes[2],
    )

    if title:
        fig.suptitle(
            title,
            y=1.02,
        )

    plt.tight_layout()
    plt.show()


## True model

In [ ]:
true_parameters = {
    "flux": 1000.0,
    "re": 8.0,
    "n": 2.0,
    "q": 0.8,
    "theta": 0.0,
    "x0": 30.0,
    "y0": 30.0,
}

true_model = SersicModel(
    **true_parameters,
)

## Render mock image

In [ ]:
data = true_model.render(
    shape=(60, 60),
)

noise = make_constant_noise_map(
    data.shape,
    sigma=1.0,
)

In [ ]:
show_log_image(
    data,
    title="Single-Sérsic mock",
)

print("Total rendered flux:", data.sum())

## Model factory

In [ ]:
def model_factory(
    flux,
    re,
    n,
):
    return SersicModel(
        flux=flux,
        re=re,
        n=n,
        q=true_parameters["q"],
        theta=true_parameters["theta"],
        x0=true_parameters["x0"],
        y0=true_parameters["y0"],
    )

## ParameterSet

In [ ]:
parameters = ParameterSet(
    [
        Parameter(
            "flux",
            100.0,
            2000.0,
        ),
        Parameter(
            "re",
            2.0,
            20.0,
        ),
        Parameter(
            "n",
            0.5,
            8.0,
        ),
    ]
)

## Fit

In [ ]:
fitter = DynestyFitter(
    model_factory=model_factory,
    data=data,
    noise=noise,
    parameters=parameters,
)

result = fitter.run(
    nlive=150,
    sample="rwalk",
    print_progress=True,
)

## Posterior summaries

In [ ]:
for name in parameters.names:
    print("True", name + ":", true_parameters[name])
    print(result.summary(name))
    print()

## Best-likelihood model and residual

In [ ]:
best_parameters = result.best_parameters
best_parameters

In [ ]:
plot_log_data_model_residual(
    data,
    best_image,
    title="Single-Sérsic best fit",
)

## Recovery check

In [ ]:
for name in parameters.names:
    true_value = true_parameters[name]
    recovered_value = best_parameters[name]
    error_percent = 100 * (recovered_value - true_value) / true_value

    print(f"{name}: true={true_value:.6g}, recovered={recovered_value:.6g}, error={error_percent:.3g}%")